In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" dire1ctory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sachinsingh152/mine11/bert_ready_data.csv


In [2]:
# ============================================
# 📌 1. IMPORTS
# ============================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Running on: {device}")

# ============================================
# 📌 2. DATA PREPARATION
# ============================================
DATA_PATH = "/kaggle/input/datasets/sachinsingh152/mine11/bert_ready_data.csv"

# Load and clean data
df = pd.read_csv(DATA_PATH)

# Ensure labels are integers
if df['label'].dtype == "object":
    df['label'] = df['label'].map({'negative': 0, 'positive': 1})

# Split data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# ============================================
# 📌 3. TOKENIZER & DATASET
# ============================================
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
MAX_LEN = 128  # Increased to 128 for better accuracy; change to 64 if OOM occurs

class DatasetClass(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create DataLoaders
train_loader = DataLoader(
    DatasetClass(train_df['text'], train_df['label'], tokenizer, MAX_LEN), 
    batch_size=16, 
    shuffle=True
)
test_loader = DataLoader(
    DatasetClass(test_df['text'], test_df['label'], tokenizer, MAX_LEN), 
    batch_size=16
)

# ============================================
# 📌 4. MODEL ARCHITECTURE (SEMI-FROZEN)
# ============================================
class FastModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        
        # Freeze all layers first
        for param in self.bert.parameters():
            param.requires_grad = False
            
        # 🔥 UNFREEZE the last transformer layer to improve score significantly
        for param in self.bert.transformer.layer[-1].parameters():
            param.requires_grad = True

        # Custom Classifier Head
        self.fc = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state[:, 0]  # Take CLS token representation
        return self.fc(x)

model = FastModel().to(device)

# Using AdamW with a lower learning rate for the unfrozen layer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
loss_fn = nn.CrossEntropyLoss()

# ============================================
# 📌 5. TRAINING LOOP
# ============================================
EPOCHS = 4 

print("\n--- Training Started ---")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, mask)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"✅ Epoch {epoch+1} Avg Loss: {avg_loss:.4f}")

# ============================================
# 📌 6. EVALUATION
# ============================================
model.eval()
preds, y_true = [], []

print("\n--- Running Evaluation ---")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label']

        outputs = model(input_ids, mask)
        predictions = torch.argmax(outputs, dim=1)

        preds.extend(predictions.cpu().numpy())
        y_true.extend(labels.numpy())

print(f"\n🎯 Final Accuracy: {accuracy_score(y_true, preds):.4f}")
print("\nDetailed Report:")
print(classification_report(y_true, preds))

# ============================================
# 📌 7. SAVE FOR DOWNLOAD
# ============================================
SAVE_PATH = "/kaggle/working/fast_sentiment_model"
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

torch.save(model.state_dict(), os.path.join(SAVE_PATH, "model_weights.pth"))
tokenizer.save_pretrained(SAVE_PATH)
print(f"\n💾 Model saved to: {SAVE_PATH}")

🚀 Running on: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

✅ Epoch 2 Avg Loss: 0.5894


Epoch 3/4: 100%|██████████| 38013/38013 [51:23<00:00, 12.33it/s, loss=0.423]


✅ Epoch 3 Avg Loss: 0.5665


Epoch 4/4: 100%|██████████| 38013/38013 [51:23<00:00, 12.33it/s, loss=0.593]


✅ Epoch 4 Avg Loss: 0.5469

--- Running Evaluation ---

🎯 Final Accuracy: 0.6749

Detailed Report:
              precision    recall  f1-score   support

           0       0.68      0.66      0.67     75776
           1       0.67      0.69      0.68     76275

    accuracy                           0.67    152051
   macro avg       0.68      0.67      0.67    152051
weighted avg       0.68      0.67      0.67    152051


💾 Model saved to: /kaggle/working/fast_sentiment_model
